In [2]:
import arcpy
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional, Union
import os


In [3]:
streets_network = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\Streets_Network_Tahoe'

In [4]:
# print streets network fields
street_fields = [f.name for f in arcpy.ListFields(streets_network)]
print(street_fields)

['OBJECTID', 'Shape', 'segment_id', 'from_connector', 'to_connector', 'class', 'name', 'surface', 'speed_limit_kmh', 'length_m', 'DriveTime', 'In_Basin', 'created_user', 'created_date', 'last_edited_user', 'last_edited_date', 'in_flood_zone', 'high_sev_fire', 'landslide_mean', 'landslide_max', 'Below_Poverty_Household_max', 'Speak_English_Not_Well_Not_at_A_max', 'Vehicle_Available_0_max', 'With_Disability_max', 'diff_length_max', 'detour_max', 'slr_diff_length_max', 'slr_detour_max', 'Estimated_2024_AADT', 'aadt_zone_name', 'Shape_Length', 'Estimated_2024_AADT_Filled', 'slr_diff_length_max_from_neighbors', 'AADT_Jenks_Score', 'SLR_Diff_Length_Jenks_Score', 'Equity_Score', 'Equity_Score_Normalized', 'Below_Poverty_Household_max_Normalized', 'Speak_English_Not_Well_Not_at_A_max_Normalized', 'Vehicle_Available_0_max_Normalized', 'With_Disability_max_Normalized', 'traffic_evacuation', 'egalitarian_evacuation', 'prioritarian_evacuation', 'routed_evacuation', 'par_alt_evacuation', 'traffic_l

In [19]:

# ============================================================================
# SECTION 3: NORMALIZATION FUNCTIONS
# ============================================================================

def calculate_zscore(
    values: pd.Series,
    mean: Optional[float] = None,
    std: Optional[float] = None
) -> Tuple[pd.Series, float, float]:
    """
    Calculate z-scores for a series of values.
    
    Parameters:
    -----------
    values : pd.Series
        Input values
    mean : float, optional
        Custom mean (if None, calculated from values)
    std : float, optional
        Custom standard deviation (if None, calculated from values)
    
    Returns:
    --------
    tuple
        (z_scores, mean, std)
    """
    if mean is None:
        mean = values.mean()
    if std is None:
        std = values.std()
    
    if std == 0:
        print(f"Warning: Standard deviation is 0. Returning zeros.")
        return pd.Series([0] * len(values), index=values.index), mean, std
    
    z_scores = (values - mean) / std
    return z_scores, mean, std


def calculate_minmax(
    values: pd.Series,
    min_val: Optional[float] = None,
    max_val: Optional[float] = None
) -> Tuple[pd.Series, float, float]:
    """
    Calculate min-max normalized values (0-1 scale).
    
    Parameters:
    -----------
    values : pd.Series
        Input values
    min_val : float, optional
        Custom minimum (if None, calculated from values)
    max_val : float, optional
        Custom maximum (if None, calculated from values)
    
    Returns:
    --------
    tuple
        (normalized_values, min, max)
    """
    if min_val is None:
        min_val = values.min()
    if max_val is None:
        max_val = values.max()
    
    if max_val == min_val:
        print(f"Warning: Range is 0. Returning zeros.")
        return pd.Series([0] * len(values), index=values.index), min_val, max_val
    
    normalized = (values - min_val) / (max_val - min_val)
    return normalized, min_val, max_val


# ============================================================================
# SECTION 4: COMPOSITE INDEX CALCULATION
# ============================================================================

def load_config_csv(config_csv: str) -> pd.DataFrame:
    """
    Load configuration CSV for index calculation.
    
    Expected columns:
    - field_name: Name of the field to include
    - weight: Weight for this field within its subindex
    - invert: 1 to invert (multiply by -1), 0 otherwise
    - subindex: Name of the subindex this field belongs to
    
    Parameters:
    -----------
    config_csv : str
        Path to configuration CSV file
    
    Returns:
    --------
    pd.DataFrame
        Configuration data
    """
    config = pd.read_csv(config_csv)
    
    # Validate required columns
    required_cols = ['field_name', 'weight', 'invert', 'subindex']
    missing_cols = [col for col in required_cols if col not in config.columns]
    
    if missing_cols:
        raise ValueError(f"Missing required columns in config CSV: {missing_cols}")
    
    # Validate weights sum to 1.0 within each subindex
    weight_sums = config.groupby('subindex')['weight'].sum()
    for subindex, weight_sum in weight_sums.items():
        if not np.isclose(weight_sum, 1.0, atol=0.001):
            print(f"Warning: Weights for subindex '{subindex}' sum to {weight_sum:.3f}, not 1.0")
    
    return config


def load_custom_params(custom_params_csv: str) -> Optional[pd.DataFrame]:
    """
    Load custom normalization parameters from CSV.
    
    Expected columns:
    - field_name: Name of the field
    - mean (for z-score) or min (for minmax)
    - std (for z-score) or max (for minmax)
    
    Parameters:
    -----------
    custom_params_csv : str
        Path to custom parameters CSV
    
    Returns:
    --------
    pd.DataFrame or None
        Custom parameters if file exists, None otherwise
    """
    if custom_params_csv and os.path.exists(custom_params_csv):
        params = pd.read_csv(custom_params_csv)
        print(f"Loaded custom normalization parameters for {len(params)} fields")
        return params
    return None


def normalize_field(
    df: pd.DataFrame,
    field_name: str,
    normalization: str = 'zscore',
    invert: bool = False,
    custom_params: Optional[pd.DataFrame] = None
) -> Tuple[pd.Series, Dict[str, float]]:
    """
    Normalize a single field using z-score or min-max normalization.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    field_name : str
        Name of field to normalize
    normalization : str
        'zscore' or 'minmax'
    invert : bool
        Whether to invert the values (multiply by -1)
    custom_params : pd.DataFrame, optional
        Custom normalization parameters
    
    Returns:
    --------
    tuple
        (normalized_series, params_dict)
    """
    values = df[field_name].copy()
    
    # Get custom parameters if available
    mean_val, std_val = None, None
    min_val, max_val = None, None
    
    if custom_params is not None:
        param_row = custom_params[custom_params['field_name'] == field_name]
        if not param_row.empty:
            if normalization == 'zscore':
                mean_val = param_row.iloc[0].get('mean', None)
                std_val = param_row.iloc[0].get('std', None)
            else:
                min_val = param_row.iloc[0].get('min', None)
                max_val = param_row.iloc[0].get('max', None)
    
    # Perform normalization
    if normalization == 'zscore':
        normalized, mean_val, std_val = calculate_zscore(values, mean_val, std_val)
        params = {'mean': mean_val, 'std': std_val}
    elif normalization == 'minmax':
        normalized, min_val, max_val = calculate_minmax(values, min_val, max_val)
        params = {'min': min_val, 'max': max_val}
    else:
        raise ValueError(f"Unknown normalization method: {normalization}")
    
    # Invert if specified
    if invert:
        normalized = normalized * -1
    
    return normalized, params


def calculate_subindex(
    df: pd.DataFrame,
    config: pd.DataFrame,
    subindex_name: str,
    normalization: str = 'zscore',
    custom_params: Optional[pd.DataFrame] = None
) -> Tuple[pd.Series, pd.DataFrame]:
    """
    Calculate a single subindex from weighted normalized fields.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with raw values
    config : pd.DataFrame
        Configuration for this subindex (filtered)
    subindex_name : str
        Name of the subindex
    normalization : str
        Normalization method ('zscore' or 'minmax')
    custom_params : pd.DataFrame, optional
        Custom normalization parameters
    
    Returns:
    --------
    tuple
        (subindex_series, normalization_params_df)
    """
    print(f"\nCalculating subindex: {subindex_name}")
    
    subindex_value = pd.Series([0.0] * len(df), index=df.index)
    norm_params_list = []
    
    for _, row in config.iterrows():
        field_name = row['field_name']
        weight = row['weight']
        invert = bool(row['invert'])
        
        if field_name not in df.columns:
            print(f"  Warning: Field '{field_name}' not found in dataframe. Skipping.")
            continue
        
        # Normalize the field
        normalized, params = normalize_field(
            df, field_name, normalization, invert, custom_params
        )
        
        # Add to subindex
        subindex_value += normalized * weight
        
        # Store normalization parameters
        params['field_name'] = field_name
        params['weight'] = weight
        params['invert'] = int(invert)
        params['subindex'] = subindex_name
        norm_params_list.append(params)
        
        print(f"  {field_name}: weight={weight}, invert={invert}")
    
    norm_params_df = pd.DataFrame(norm_params_list)
    
    return subindex_value, norm_params_df


def calculate_composite_index(
    feature_class: str,
    config_csv: str,
    output_feature_class: str,
    output_field: str = "Mobility_Index",
    normalization: str = 'zscore',
    workspace: Optional[str] = None,
    custom_params_csv: Optional[str] = None,
    export_params: bool = True
) -> pd.DataFrame:
    """
    Calculate a composite index from multiple weighted subindices.
    
    This is the main function that orchestrates the entire index calculation process.
    
    Parameters:
    -----------
    feature_class : str
        Input feature class name (or full path if workspace not provided)
    config_csv : str
        Path to configuration CSV
    output_feature_class : str
        Output feature class name
    output_field : str
        Name of the final index field
    normalization : str
        Normalization method: 'zscore' or 'minmax'
    workspace : str, optional
        Workspace path (geodatabase)
    custom_params_csv : str, optional
        Path to custom normalization parameters CSV
    export_params : bool
        Whether to export normalization parameters to CSV
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with all calculated indices and subindices
    
    Example CSV config file:
    ------------------------
    field_name,weight,invert,subindex
    Point_Count,0.5,0,Access
    EMPNUM_Sum,0.5,0,Access
    Max_Number_Routes,0.6,0,Transit
    Bus_Frequency,0.4,0,Transit
    Min_Dist_Sidewalk,1.0,1,Walkability
    """
    # Set workspace if provided
    if workspace:
        arcpy.env.workspace = workspace
        fc_path = os.path.join(workspace, feature_class)
        output_fc_path = os.path.join(workspace, output_feature_class)
    else:
        fc_path = feature_class
        output_fc_path = output_feature_class
    
    print("="*60)
    print("COMPOSITE INDEX CALCULATION")
    print("="*60)
    print(f"Input feature class: {fc_path}")
    print(f"Output feature class: {output_fc_path}")
    print(f"Normalization method: {normalization}")
    print(f"Configuration: {config_csv}")
    
    # Load configuration
    config = load_config_csv(config_csv)
    print(f"\nLoaded configuration with {len(config)} fields across {config['subindex'].nunique()} subindices")
    
    # Load custom parameters if provided
    custom_params = load_custom_params(custom_params_csv)
    
    # Read feature class into DataFrame
    print("\nReading feature class...")
    fields_to_read = ['GRID_ID'] + config['field_name'].tolist()
    
    # Check which fields exist
    existing_fields = [f.name for f in arcpy.ListFields(fc_path)]
    fields_to_read = [f for f in fields_to_read if f in existing_fields]
    
    data = []
    with arcpy.da.SearchCursor(fc_path, fields_to_read) as cursor:
        for row in cursor:
            data.append(row)
    
    df = pd.DataFrame(data, columns=fields_to_read)
    print(f"Read {len(df)} features")
    
    # Calculate each subindex
    subindices = {}
    all_norm_params = []
    
    for subindex_name in config['subindex'].unique():
        subindex_config = config[config['subindex'] == subindex_name]
        subindex_value, norm_params = calculate_subindex(
            df, subindex_config, subindex_name, normalization, custom_params
        )
        subindices[subindex_name] = subindex_value
        all_norm_params.append(norm_params)
        
        # Add subindex to dataframe
        df[f'{subindex_name}_Index'] = subindex_value
    
    # Combine all normalization parameters
    variable_params = pd.concat(all_norm_params, ignore_index=True)
    
    # Normalize each subindex
    print("\n" + "="*60)
    print("NORMALIZING SUBINDICES")
    print("="*60)
    
    subindex_params_list = []
    
    for subindex_name in subindices.keys():
        # Check if custom params exist for this subindex
        subindex_mean, subindex_std = None, None
        subindex_min, subindex_max = None, None
        
        if custom_params is not None:
            param_row = custom_params[custom_params['field_name'] == f'{subindex_name}_Index']
            if not param_row.empty:
                if normalization == 'zscore':
                    subindex_mean = param_row.iloc[0].get('mean', None)
                    subindex_std = param_row.iloc[0].get('std', None)
                else:
                    subindex_min = param_row.iloc[0].get('min', None)
                    subindex_max = param_row.iloc[0].get('max', None)
        
        # Normalize subindex
        if normalization == 'zscore':
            normalized, subindex_mean, subindex_std = calculate_zscore(
                subindices[subindex_name], subindex_mean, subindex_std
            )
            params = {'field_name': f'{subindex_name}_Index', 'mean': subindex_mean, 'std': subindex_std}
        else:
            normalized, subindex_min, subindex_max = calculate_minmax(
                subindices[subindex_name], subindex_min, subindex_max
            )
            params = {'field_name': f'{subindex_name}_Index', 'min': subindex_min, 'max': subindex_max}
        
        df[f'{subindex_name}_Index_Scaled'] = normalized
        subindex_params_list.append(params)
        
        print(f"{subindex_name}_Index: {params}")
    
    # Calculate final composite index (equal weight for all subindices)
    print("\n" + "="*60)
    print("CALCULATING FINAL COMPOSITE INDEX")
    print("="*60)
    
    num_subindices = len(subindices)
    composite_index = pd.Series([0.0] * len(df), index=df.index)
    
    for subindex_name in subindices.keys():
        composite_index += df[f'{subindex_name}_Index_Scaled'] / num_subindices
    
    df[output_field] = composite_index
    
    print(f"Final {output_field} calculated using {num_subindices} equally-weighted subindices")
    print(f"  Mean: {composite_index.mean():.4f}")
    print(f"  Std: {composite_index.std():.4f}")
    print(f"  Min: {composite_index.min():.4f}")
    print(f"  Max: {composite_index.max():.4f}")
    
    # Export normalization parameters
    if export_params:
        subindex_params_df = pd.DataFrame(subindex_params_list)
        
        # Export variable-level parameters
        variable_params_path = config_csv.replace('.csv', '_normalization_params.csv')
        variable_params.to_csv(variable_params_path, index=False)
        print(f"\nVariable normalization parameters exported to: {variable_params_path}")
        
        # Export subindex-level parameters
        subindex_params_path = config_csv.replace('.csv', '_subindex_normalization_params.csv')
        subindex_params_df.to_csv(subindex_params_path, index=False)
        print(f"Subindex normalization parameters exported to: {subindex_params_path}")
    
    # Copy feature class and add results
    print("\n" + "="*60)
    print("WRITING RESULTS TO FEATURE CLASS")
    print("="*60)
    
    if arcpy.Exists(output_fc_path):
        arcpy.Delete_management(output_fc_path)
        print(f"Deleted existing output: {output_fc_path}")
    
    arcpy.Copy_management(fc_path, output_fc_path)
    print(f"Copied input to output: {output_fc_path}")
    
    # Add new fields
    existing_output_fields = [f.name for f in arcpy.ListFields(output_fc_path)]
    
    fields_to_add = []
    for col in df.columns:
        if col not in existing_output_fields and col != 'GRID_ID':
            fields_to_add.append(col)
    
    for field in fields_to_add:
        arcpy.AddField_management(output_fc_path, field, 'DOUBLE')
        print(f"  Added field: {field}")
    
    # Update values
    update_fields = ['GRID_ID'] + fields_to_add
    update_dict = df.set_index('GRID_ID')[fields_to_add].to_dict('index')
    
    print("\nUpdating feature class values...")
    with arcpy.da.UpdateCursor(output_fc_path, update_fields) as cursor:
        for row in cursor:
            grid_id = row[0]
            if grid_id in update_dict:
                for i, field in enumerate(fields_to_add):
                    row[i + 1] = update_dict[grid_id][field]
                cursor.updateRow(row)
    
    print("Feature class update complete!")
    print("="*60)
    
    return df


In [20]:
# ============================================================================
# SECTION 5: EQUITY INDEX CALCULATION
# ============================================================================

def calculate_equity_index(
    df: pd.DataFrame,
    normalization: str = 'zscore',
    custom_params: Optional[pd.DataFrame] = None
) -> Tuple[pd.Series, pd.DataFrame]:
    """
    Calculate Equity Index from four equally-weighted demographic fields.

    Fields with equal weight (0.25 each):
    - Below_Poverty_Household_max
    - Speak_English_Not_Well_Not_at_A_max
    - Vehicle_Available_0_max
    - With_Disability_max

    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with raw values
    normalization : str
        Normalization method ('zscore' or 'minmax')
    custom_params : pd.DataFrame, optional
        Custom normalization parameters

    Returns:
    --------
    tuple
        (equity_index_series, normalization_params_df)
    
    Example:
    --------
    equity_index, equity_params = calculate_equity_index(df)
    df['Equity_Index'] = equity_index
    """
    # Create config for Equity Index with equal weights (0.25 each)
    equity_config = pd.DataFrame([
        {'field_name': 'Below_Poverty_Household_max', 'weight': 0.25, 'invert': 0, 'subindex': 'Equity'},
        {'field_name': 'Speak_English_Not_Well_Not_at_A_max', 'weight': 0.25, 'invert': 0, 'subindex': 'Equity'},
        {'field_name': 'Vehicle_Available_0_max', 'weight': 0.25, 'invert': 0, 'subindex': 'Equity'},
        {'field_name': 'With_Disability_max', 'weight': 0.25, 'invert': 0, 'subindex': 'Equity'},
    ])

    equity_index, norm_params = calculate_subindex(
        df, equity_config, 'Equity', normalization, custom_params
    )

    return equity_index, norm_params


def calculate_equity_index_feature_class(
    feature_class: str,
    workspace: Optional[str] = None
) -> Dict[str, float]:
    """
    Calculate Equity Index for a feature class and populate all fields.

    Creates and populates the following fields:
    - Equity_Score (raw composite z-score)
    - Equity_Score_Normalized (0-1 scale)
    - Below_Poverty_Household_max_Normalized (0-1 scale)
    - Speak_English_Not_Well_Not_at_A_max_Normalized (0-1 scale)
    - Vehicle_Available_0_max_Normalized (0-1 scale)
    - With_Disability_max_Normalized (0-1 scale)

    Parameters:
    -----------
    feature_class : str
        Name of feature class (or full path if workspace not provided)
    workspace : str, optional
        Workspace path (geodatabase)

    Returns:
    --------
    dict
        Statistics including min/max for equity score and normalized values

    Example:
    --------
    stats = calculate_equity_index_feature_class(
        'Streets_Network_Tahoe',
        workspace=r'F:\GIS\PROJECTS\...\PROTECT_analysis.gdb'
    )
    print(f"Equity score range: {stats['equity_score_min']:.4f} to {stats['equity_score_max']:.4f}")
    """
    # Set workspace if provided
    if workspace:
        arcpy.env.workspace = workspace
        fc_path = os.path.join(workspace, feature_class)
    else:
        fc_path = feature_class

    print("="*60)
    print("EQUITY INDEX CALCULATION")
    print("="*60)
    print(f"Feature class: {fc_path}")

    # Equity field configuration
    equity_fields = [
        'Below_Poverty_Household_max',
        'Speak_English_Not_Well_Not_at_A_max',
        'Vehicle_Available_0_max',
        'With_Disability_max'
    ]

    # Validate input fields exist
    existing_fields = [f.name for f in arcpy.ListFields(fc_path)]
    missing_fields = [f for f in equity_fields if f not in existing_fields]
    if missing_fields:
        raise ValueError(f"Missing equity fields in {fc_path}: {missing_fields}")

    # Read data from feature class
    print("\nReading feature class...")
    data = []
    oid_field = arcpy.Describe(fc_path).OIDFieldName
    read_fields = [oid_field] + equity_fields

    with arcpy.da.SearchCursor(fc_path, read_fields) as cursor:
        for row in cursor:
            data.append(row)

    df = pd.DataFrame(data, columns=read_fields)
    print(f"Read {len(df)} features")

    # Calculate normalized values for each input field (0-1 scale)
    print("\nCalculating normalized input fields...")
    normalized_fields = {}
    for field in equity_fields:
        normalized, min_val, max_val = calculate_minmax(df[field])
        normalized_fields[f'{field}_Normalized'] = normalized
        print(f"  {field}: min={min_val:.4f}, max={max_val:.4f}")

    # Calculate raw equity index (z-score normalization)
    print("\nCalculating raw Equity Index (z-score)...")
    equity_index, equity_params = calculate_equity_index(df, normalization='zscore')
    df['Equity_Score'] = equity_index

    print(f"  Mean: {equity_index.mean():.4f}")
    print(f"  Std: {equity_index.std():.4f}")
    print(f"  Min: {equity_index.min():.4f}")
    print(f"  Max: {equity_index.max():.4f}")

    # Normalize equity score to 0-1 scale
    print("\nNormalizing Equity Index to 0-1 scale...")
    equity_normalized, eq_min, eq_max = calculate_minmax(equity_index)
    df['Equity_Score_Normalized'] = equity_normalized
    print(f"  Normalized range: {eq_min:.4f} to {eq_max:.4f}")

    # Add normalized input fields to dataframe
    for field_name, normalized_values in normalized_fields.items():
        df[field_name] = normalized_values

    # Create output fields if they don't exist
    print("\nAdding fields to feature class...")
    all_output_fields = [
        'Equity_Score',
        'Equity_Score_Normalized',
        *[f'{field}_Normalized' for field in equity_fields]
    ]

    for field_name in all_output_fields:
        if field_name in existing_fields:
            print(f"  {field_name}: already exists")
        else:
            arcpy.AddField_management(fc_path, field_name, 'DOUBLE')
            print(f"  {field_name}: created")

    # Populate fields
    print("\nPopulating fields...")
    update_fields = [oid_field] + all_output_fields
    update_dict = df.set_index(oid_field)[all_output_fields].to_dict('index')

    update_count = 0
    with arcpy.da.UpdateCursor(fc_path, update_fields) as cursor:
        for row in cursor:
            oid = row[0]
            if oid in update_dict:
                for i, field in enumerate(all_output_fields):
                    row[i + 1] = update_dict[oid][field]
                cursor.updateRow(row)
                update_count += 1

    print(f"Updated {update_count} features")
    print("="*60)

    # Return statistics
    stats = {
        'equity_score_min': df['Equity_Score'].min(),
        'equity_score_max': df['Equity_Score'].max(),
        'equity_score_mean': df['Equity_Score'].mean(),
        'equity_score_normalized_min': df['Equity_Score_Normalized'].min(),
        'equity_score_normalized_max': df['Equity_Score_Normalized'].max(),
    }

    return stats

<>:84: SyntaxWarning: invalid escape sequence '\G'
<>:84: SyntaxWarning: invalid escape sequence '\G'
C:\Users\amcclary\AppData\Local\Temp\ipykernel_45128\797451886.py:84: SyntaxWarning: invalid escape sequence '\G'
  workspace=r'F:\GIS\PROJECTS\...\PROTECT_analysis.gdb'


In [22]:
# Example: Calculate Equity Index and populate feature class with all fields
arcpy.env.workspace = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis'
gdb = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb'

# Calculate equity index and populate all fields
stats = calculate_equity_index_feature_class(
    feature_class='Streets_Network_Tahoe',
    workspace=gdb
)

# Display results
print(f"\nEquity Index Statistics:")
print(f"  Raw score range: {stats['equity_score_min']:.4f} to {stats['equity_score_max']:.4f}")
print(f"  Raw score mean: {stats['equity_score_mean']:.4f}")
print(f"  Normalized score range: {stats['equity_score_normalized_min']:.4f} to {stats['equity_score_normalized_max']:.4f}")

EQUITY INDEX CALCULATION
Feature class: F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\Streets_Network_Tahoe

Reading feature class...
Read 18285 features

Calculating normalized input fields...
  Below_Poverty_Household_max: min=0.0000, max=142.0000
  Speak_English_Not_Well_Not_at_A_max: min=0.0000, max=196.0000
  Vehicle_Available_0_max: min=0.0000, max=92.0000
  With_Disability_max: min=0.0000, max=257.0000

Calculating raw Equity Index (z-score)...

Calculating subindex: Equity
  Below_Poverty_Household_max: weight=0.25, invert=False
  Speak_English_Not_Well_Not_at_A_max: weight=0.25, invert=False
  Vehicle_Available_0_max: weight=0.25, invert=False
  With_Disability_max: weight=0.25, invert=False
  Mean: -0.0000
  Std: 0.7714
  Min: -0.6886
  Max: 4.0575

Normalizing Equity Index to 0-1 scale...
  Normalized range: -0.6886 to 4.0575

Adding fields to feature class...
  Equity_Score: created
  Equity_Score_Normalized: created
  Below_Poverty_Household_

In [12]:
# ============================================================================
# SECTION 6: JENKS NATURAL BREAKS CLASSIFICATION
# ============================================================================

def classify_with_jenks(
    values: pd.Series,
    n_categories: int = 10
) -> Tuple[pd.Series, List[float]]:
    """
    Classify values using Jenks natural breaks classification.

    Assigns a numerical score (1 to n_categories) based on Jenks natural breaks.
    Null values are assigned a score of 1.

    Parameters:
    -----------
    values : pd.Series
        Input values to classify
    n_categories : int
        Number of categories/classes (default 10). Must be between 2 and len(unique_values).

    Returns:
    --------
    tuple
        (scores_series, break_points_list)
        - scores_series: Series with classification scores (1 to n_categories)
        - break_points_list: List of break points from Jenks classification

    Example:
    --------
    scores, breaks = classify_with_jenks(df['some_field'], n_categories=10)
    df['some_field_score'] = scores
    print(f"Break points: {breaks}")
    """
    try:
        import jenkspy
    except ImportError:
        raise ImportError("jenkspy is required for Jenks classification. Install with: pip install jenkspy")

    # Create output series with nulls initially set to 1
    scores = pd.Series([1] * len(values), index=values.index, dtype=int)

    # Get non-null values for classification
    non_null_mask = values.notna()
    non_null_values = values[non_null_mask].dropna()

    if len(non_null_values) == 0:
        print("Warning: All values are null. Returning all scores as 1.")
        return scores, []

    # Handle case where n_categories is greater than unique values
    unique_count = non_null_values.nunique()
    if n_categories > unique_count:
        print(f"Warning: n_categories ({n_categories}) > unique values ({unique_count}). Using {unique_count} categories.")
        n_categories = unique_count

    # Calculate Jenks breaks
    breaks = jenkspy.jenks_breaks(non_null_values.values, n_classes=n_categories)

    # Assign scores based on which break interval each value falls into
    # breaks has n_categories + 1 elements (start and end points)
    for idx, val in zip(non_null_values.index, non_null_values.values):
        # Find which break interval this value belongs to
        for i in range(len(breaks) - 1):
            if i == len(breaks) - 2:  # Last interval (inclusive on both ends)
                if breaks[i] <= val <= breaks[i + 1]:
                    scores[idx] = i + 1
                    break
            else:  # Other intervals
                if breaks[i] <= val < breaks[i + 1]:
                    scores[idx] = i + 1
                    break

    return scores, breaks


def classify_feature_class_with_jenks(
    feature_class: str,
    input_field: str,
    output_field: str,
    n_categories: int = 10,
    workspace: Optional[str] = None
) -> Tuple[List[float], Dict[int, int]]:
    """
    Classify a feature class field using Jenks natural breaks and populate new field.

    Creates a new field in the feature class and populates it with Jenks classification scores.

    Parameters:
    -----------
    feature_class : str
        Name of feature class (or full path if workspace not provided)
    input_field : str
        Name of field to classify
    output_field : str
        Name of new field to create and populate with scores
    n_categories : int
        Number of categories/classes (default 10)
    workspace : str, optional
        Workspace path (geodatabase)

    Returns:
    --------
    tuple
        (break_points, score_distribution)
        - break_points: List of Jenks break points
        - score_distribution: Dict mapping score (1 to n_categories) to count

    Example:
    --------
    breaks, distribution = classify_feature_class_with_jenks(
        'Streets_Network_Tahoe',
        'Estimated_2024_AADT_Filled',
        'AADT_Jenks_Score',
        n_categories=10,
        workspace=r'F:\GIS\PROJECTS\...\PROTECT_analysis.gdb'
    )
    print(f"Break points: {breaks}")
    print(f"Distribution: {distribution}")
    """
    # Set workspace if provided
    if workspace:
        arcpy.env.workspace = workspace
        fc_path = os.path.join(workspace, feature_class)
    else:
        fc_path = feature_class

    print("="*60)
    print(f"JENKS CLASSIFICATION: {input_field} → {output_field}")
    print("="*60)
    print(f"Feature class: {fc_path}")
    print(f"Input field: {input_field}")
    print(f"Output field: {output_field}")
    print(f"Categories: {n_categories}")

    # Validate input field exists
    existing_fields = [f.name for f in arcpy.ListFields(fc_path)]
    if input_field not in existing_fields:
        raise ValueError(f"Field '{input_field}' not found in {fc_path}")

    # Read data from feature class
    print("\nReading feature class...")
    data = []
    oid_field = arcpy.Describe(fc_path).OIDFieldName
    read_fields = [oid_field, input_field]

    with arcpy.da.SearchCursor(fc_path, read_fields) as cursor:
        for row in cursor:
            data.append(row)

    df = pd.DataFrame(data, columns=[oid_field, input_field])
    print(f"Read {len(df)} features")

    # Apply Jenks classification
    print(f"\nApplying Jenks classification...")
    scores, breaks = classify_with_jenks(df[input_field], n_categories=n_categories)

    print(f"Break points: {breaks}")
    print(f"\nScore distribution:")
    score_dist = scores.value_counts().sort_index().to_dict()
    for score, count in score_dist.items():
        print(f"  Score {score}: {count} features")

    # Create output field if it doesn't exist
    print(f"\nAdding field '{output_field}'...")
    if output_field in existing_fields:
        print(f"  Field already exists. Skipping creation.")
    else:
        arcpy.AddField_management(fc_path, output_field, 'SHORT')
        print(f"  Field created successfully.")

    # Populate the field
    print(f"Populating '{output_field}'...")
    update_fields = [oid_field, output_field]
    score_dict = dict(zip(df[oid_field], scores))

    update_count = 0
    with arcpy.da.UpdateCursor(fc_path, update_fields) as cursor:
        for row in cursor:
            oid = row[0]
            if oid in score_dict:
                row[1] = int(score_dict[oid])
                cursor.updateRow(row)
                update_count += 1

    print(f"Updated {update_count} features")
    print("="*60)

    return breaks, score_dist

<>:116: SyntaxWarning: invalid escape sequence '\G'
<>:116: SyntaxWarning: invalid escape sequence '\G'
C:\Users\amcclary\AppData\Local\Temp\ipykernel_45128\2018713114.py:116: SyntaxWarning: invalid escape sequence '\G'
  workspace=r'F:\GIS\PROJECTS\...\PROTECT_analysis.gdb'


In [13]:
# Example: Apply Jenks classification directly to feature class and populate new field
arcpy.env.workspace = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis'
gdb = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb'

# Classify AADT field and create/populate AADT_Jenks_Score field
breaks, distribution = classify_feature_class_with_jenks(
    feature_class='Streets_Network_Tahoe',
    input_field='Estimated_2024_AADT_Filled',
    output_field='AADT_Jenks_Score',
    n_categories=10,
    workspace=gdb
)

# Verify the results
print(f"\nBreak points: {breaks}")
print(f"Score distribution: {distribution}")

JENKS CLASSIFICATION: Estimated_2024_AADT_Filled → AADT_Jenks_Score
Feature class: F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\Streets_Network_Tahoe
Input field: Estimated_2024_AADT_Filled
Output field: AADT_Jenks_Score
Categories: 10

Reading feature class...
Read 18285 features

Applying Jenks classification...
Break points: [np.float64(0.0), np.float64(553.0), np.float64(1918.0), np.float64(3914.0), np.float64(6091.0), np.float64(9025.0), np.float64(12406.0), np.float64(15446.0), np.float64(22554.0), np.float64(26727.0), np.float64(31624.0)]

Score distribution:
  Score 1: 15802 features
  Score 2: 686 features
  Score 3: 393 features
  Score 4: 222 features
  Score 5: 477 features
  Score 6: 307 features
  Score 7: 151 features
  Score 8: 20 features
  Score 9: 149 features
  Score 10: 78 features

Adding field 'AADT_Jenks_Score'...
  Field created successfully.
Populating 'AADT_Jenks_Score'...
Updated 18285 features

Break points: [np.float64(0.0),

In [14]:
# Example: Apply Jenks classification directly to feature class and populate new field
arcpy.env.workspace = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis'
gdb = r'F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb'

# Classify AADT field and create/populate AADT_Jenks_Score field
breaks, distribution = classify_feature_class_with_jenks(
    feature_class='Streets_Network_Tahoe',
    input_field='slr_diff_length_max',
    output_field='SLR_Diff_Length_Jenks_Score',
    n_categories=10,
    workspace=gdb
)

# Verify the results
print(f"\nBreak points: {breaks}")
print(f"Score distribution: {distribution}")

JENKS CLASSIFICATION: slr_diff_length_max → SLR_Diff_Length_Jenks_Score
Feature class: F:\GIS\PROJECTS\Transportation\Protect\PROTECT_analysis\PROTECT_analysis.gdb\Streets_Network_Tahoe
Input field: slr_diff_length_max
Output field: SLR_Diff_Length_Jenks_Score
Categories: 10

Reading feature class...
Read 18285 features

Applying Jenks classification...
Break points: [np.float64(-878.0), np.float64(696.0), np.float64(2256.0), np.float64(6893.0), np.float64(14994.0), np.float64(21357.0), np.float64(59720.0), np.float64(65730.0), np.float64(73917.0), np.float64(86067.0), np.float64(113764.0)]

Score distribution:
  Score 1: 15492 features
  Score 2: 1819 features
  Score 3: 403 features
  Score 4: 82 features
  Score 5: 21 features
  Score 6: 146 features
  Score 7: 108 features
  Score 8: 14 features
  Score 9: 48 features
  Score 10: 152 features

Adding field 'SLR_Diff_Length_Jenks_Score'...
  Field created successfully.
Populating 'SLR_Diff_Length_Jenks_Score'...
Updated 18285 featur